## Sequences

Let's try to understand which heads are most responsible for creating integer sequences

In [ ]:
%pip install transformer-lens circuitsvis torch einops tuned-lens

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 19.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.2 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=3c66d85e8fca63bde8bd9a4bf4201eb20761d772c8117bdbeb9b83de15722b69
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [ ]:
from transformer_lens import HookedTransformer, utils
import circuitsvis as cv
import torch as t
import einops

from tuned_lens.plotting import PredictionTrajectory
from tuned_lens.nn import TunedLens, Unembed, LogitLens

/tmp/ipykernel_2496/3052112055.py:1: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


In [ ]:
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small", fold_ln=False)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [ ]:
gpt2_text = "0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7,"
loss = gpt2_small(gpt2_text, return_type="loss")
print("Model loss:", loss)

Model loss: tensor(1.1333, device='cuda:0', grad_fn=<DivBackward0>)


Loss is pretty low. Must be doing an ok job...

In [ ]:
tokens = gpt2_small.to_tokens(gpt2_text)
for n in range(10):
    logits = gpt2_small(tokens)
    next_token = logits.argmax(-1, keepdim=True)[:, -1, :]
    tokens = t.concat([tokens, next_token], dim=-1)

print(gpt2_small.to_string(tokens))

['<|endoftext|>0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,']


Looks like the model can count... Let's find out why.

In [ ]:
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_text, remove_batch_dim=True)

print(gpt2_cache["pattern", 0].shape)

torch.Size([12, 27, 27])


In [ ]:
activations = t.stack([
    gpt2_cache["post", l] for l in range(gpt2_small.cfg.n_layers)
], dim=1)

cv.activations.text_neuron_activations(
    tokens=gpt2_small.to_str_tokens(gpt2_text),
    activations=activations
)

OK that didn't really tell us anything but I thought to try it anyway. It just visualised the post MLP activation magnitudes on each neuron in each layer for each token in the residual stream.

In [ ]:
rearranged_activations = utils.to_numpy(einops.rearrange(activations, "seq layers neurons -> 1 layers seq neurons"))

cv.topk_tokens.topk_tokens(
    # Some weird indexing required here ¯\_(ツ)_/¯
    tokens=[gpt2_small.to_str_tokens(gpt2_text)],
    activations=rearranged_activations,
    max_k=7,
    first_dimension_name="Layer",
    third_dimension_name="Neuron",
    first_dimension_labels=list(range(12))
)

In [ ]:
display(
    cv.attention.attention_patterns(
        tokens=gpt2_small.to_str_tokens(gpt2_text),
        attention=gpt2_cache["pattern", 1],
    )
)

Looks like L1H5 has formed "previous number" heads. Let's check if this is really the case by adding some noise.

In [ ]:
gpt2_noisy_text = "0,, 0,   0 ... 0, 0, 0 abc 1 xyz 2  ,, 3 , ... 4 hx,  5 x, 6, 7 ..."
_, gpt2_noisy_cache = gpt2_small.run_with_cache(gpt2_noisy_text, remove_batch_dim=True)

display(
    cv.attention.attention_patterns(
        tokens=gpt2_small.to_str_tokens(gpt2_noisy_text),
        attention=gpt2_noisy_cache["pattern", 1],
    )
)

## Aside

I can't tell what's going on in layer 2. In L3, there is a kind of duplicate head L3H0. But in L4H11 there seems to be our first previous token head. After this, in L5 there are several induction heads, some of which are paying attention to what comes after a previous occurrence of a pattern, or a type of token not just a previous occurence of the specific token.

I've just spent some time going through every single head in every layer. I wonder if there's a way to find the "minimum viable circuit" for a specific input -> output pair. This would be a way to identify all the "useful" attention patterns that contribute to the generation, vs the ones that don't. It should really happen across all invocations.

In [ ]:
display(
    cv.attention.attention_patterns(
        tokens=gpt2_small.to_str_tokens(gpt2_text),
        attention=gpt2_cache["pattern", 9],
    )
)

It may be useful next to see the predictions after each layer. Though MLP will make things more confusing.

## Aside

It's clear that there are layers post L1 that have more complicated patterns, but I thought the problem was already solved in L1 because of the previous number head. I suppose that may not be sufficient to generate the next token correctly. Imagine that simply transferring the information from 5, to 6, does not allow the next layer to make the leap that the next token is (6-5)+5. This is not possible because gradient descent does not have a way to get from 5 and 6 to 6-5, or to x+5. There is no "+" or "-" and even if there were, 6 and 5 are tokens represented by embeddings which may not support addition and subtraction in the sense described.

Should models be able to search this space as well? Would that make for more efficient models? Is that even a tractable problem?


## Next steps

- Visualise greedy predictions in layers - See how the prediction changes across layers.
- Formalise the definition of important attention heads so that we can detect them via code.
- Use hooks to ablate the heads that show "previous number detection" and see how that affects early and later layers.

In [ ]:
tokens = gpt2_small.to_tokens(gpt2_text)
accuracy = []
pred_accuracies = []

for l in range(gpt2_small.cfg.n_layers):
    residual_stream = gpt2_cache[f"blocks.{l}.hook_resid_post"]
    unembedded_logits = gpt2_small.unembed(residual_stream)
    predictions = unembedded_logits.argmax(-1)
    text_predictions = gpt2_small.to_str_tokens(predictions)
    
    accurate = (predictions[:-1] == tokens.squeeze()[1:])
    accuracy.append(f"L{l}: {sum(accurate).item()}")
    
    pred_accuracies.append({
        "predictions": text_predictions,
        "accuracies": t.where(accurate, 1, -1).tolist()
    })

print(accuracy)

['L0: 1', 'L1: 1', 'L2: 2', 'L3: 0', 'L4: 1', 'L5: 3', 'L6: 12', 'L7: 16', 'L8: 16', 'L9: 21', 'L10: 17', 'L11: 13']


In [ ]:
for l, acc in enumerate(pred_accuracies):
    print(f"Layer: {l}")
    display(cv.tokens.colored_tokens(acc["predictions"], acc["accuracies"]))

Layer: 0


Layer: 1


Layer: 2


Layer: 3


Layer: 4


...

Layer: 7


Layer: 8


Layer: 9


Layer: 10


Layer: 11


Not pretty, but it does the job. Raised an issue to support a similar analysis natively - https://github.com/TransformerLensOrg/CircuitsVis/issues/104.

Anyhow, we can see that in layer 6 something special happens. The model is predicting the zeros at the beginning.
In Layer 8 it learns a gew more numbers, but then in layer 9 it can predict most numbers after the zeros - it's learnt the patter. Layers 9 through to 11 then continue this.

In [ ]:
display(
    cv.attention.attention_patterns(
        tokens=gpt2_small.to_str_tokens(gpt2_text),
        attention=gpt2_cache["pattern", 9],
    )
)

We don't see anything special in L6 (gpt2_small) or L9 (gpt). But these are only the attention patterns and I wonder if there's something interesting going on in the MLP at these layers.

I also found that the logit lens is not completely accurate due to earlier layers not representing semantic information on the same basis as the final layer. So the unembed matrix may not apply to them. The latest paper for the logit lens technique is to use the tuned lens - https://tuned-lens.readthedocs.io/en/latest/tutorials/combining_with_transformer_lens.html. The tuned lens aims to

Let's quickly do that and then try to ablate some of these heads with hooks and see how that affects the result of the logit lens.

Found papers to read on similar work
- https://aclanthology.org/2024.emnlp-main.699.pdf
- https://arxiv.org/pdf/2312.09230
- https://arxiv.org/pdf/2304.14997

In particular the paper on sequences explains the role of both attention and MLP in sequence detection and recreation.

In [ ]:
import plotly.io as pio
pio.renderers.default = "sphinx_gallery" # REMOVE THIS IF YOU ARE NOT SEEING PLOTS

In [ ]:
assert gpt2_small.tokenizer is not None
tuned_lens = TunedLens.from_unembed_and_pretrained(
    unembed=Unembed(gpt2_small),
    lens_resource_id="gpt2",
).to("cuda")

logit_lens = LogitLens.from_model(gpt2_small).to("cuda")

def to_targets(input_ids: t.Tensor):
    return t.cat(
        (input_ids[..., 1:], t.full(input_ids.shape[:-1] + (1,), gpt2_small.tokenizer.eos_token_id)
    ), dim=-1)

In [ ]:
input_ids = gpt2_small.to_tokens(gpt2_text)
inputs = input_ids[:, :-1]
targets = input_ids[:, 1:]
anti_targets = t.full(inputs.shape, gpt2_small.to_single_token('<|endoftext|>'))

with t.inference_mode():
    logits, cache = gpt2_small.run_with_cache(
        input=inputs, return_type="logits"
    )

    pred_traj_clean = PredictionTrajectory.from_lens_and_cache(
        lens=tuned_lens,
        cache=cache,
        model_logits=logits,
        input_ids=inputs,
        targets=targets,
    )

    pred_traj_clean_logit = PredictionTrajectory.from_lens_and_cache(
        lens=logit_lens,
        cache=cache,
        model_logits=logits,
        input_ids=inputs,
        targets=targets,
    )

pred_traj_clean_logit.slice_sequence(slice(-20, None)).cross_entropy().clip(*(-5, 10)).figure(title="Effects of each layer on the target/anti-target ratio")


OK we can see tat most of the change happens around L9. That's really where the numbers start to come out. L10 is where the numbers are correct. The prediction in L9 is the previous number, and it seems like L10 is able to predict the correct number. So something interesting must be happeneing in the attention patterns and MLPs in those layers.

## Aside

All this is making me wonder if there is some amount of coalescing and consolidation that happens eventually with training. What are the training dynamics for these attention patterns. Can we draw conclusions about what direction they're heading in so we can tell if there are patterns that are fully formed, patterns that are more stable or more reliable or more long-lasting than others. Or are all the patterns equally stable / ephemeral.

The reason I want to know this is that it'll make it easier to spot important patterns and discard ones that are in their intermediate forms. It'll also tell us if some methods of training are more effective at bringing out certain patterns.


## Next steps

- ~~Visualise greedy predictions in layers - See how the prediction changes across layers.~~
- Formalise the definition of important attention heads so that we can detect them via code.
- Use hooks to ablate the heads that show "previous number detection" and see how that affects early and later layers.